# पाठ ०३ - एजेन्टिक डिजाइन ढाँचाहरू

यस पाठमा, हामी प्रभावकारी एआई एजेन्टहरू निर्माण गर्नका लागि तीन आधारभूत डिजाइन ढाँचाहरू अन्वेषण गर्नेछौं:

१. **स्पष्ट एजेन्ट निर्देशनहरू** — एजेन्टको व्यवहार मार्गनिर्देशन गर्ने ठ्याक्कै, भूमिका निर्धारण गर्ने प्राम्प्टहरू तयार पार्ने
२. **पायडान्टिक मोडेलहरूसँग संरचित आउटपुट** — एजेन्टहरूले अनुमान गर्न मिल्ने, प्रमाणीकरण गरिएका डाटा फर्काउने सुनिश्चित गर्ने
३. **एकल जिम्मेवारी एजेन्टहरू** — केन्द्रित एजेन्टहरू डिजाइन गर्ने जसले एक कुरा राम्रोसँग गर्दछन्

हामी प्रत्येक ढाँचालाई **यात्रा गन्तव्य सिफारिस गर्ने** परिदृश्यमा लागू गर्नेछौं, क्रमशः गन्तव्यहरू सिफारिस गर्न, उपलब्धता जाँच गर्न, र व्यवस्थापन ह्यान्डल गर्न सक्ने प्रणाली बनाउन।


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ढाँचा १: स्पष्ट एजेन्ट निर्देशनहरू

सबैभन्दा प्रभावकारी ढाँचा सबैभन्दा सरल पनि हो: तपाईंको एजेन्टका लागि स्पष्ट, विस्तृत निर्देशनहरू लेख्नु।

राम्रो निर्देशनहरूले परिभाषित गर्छ:
- **को** एजेन्ट हो (व्यक्तित्व र टोन)
- **के** गर्नुपर्छ (चरण-द्वारा-चरण जिम्मेवारीहरू)
- **कसरी** व्यवहार गर्नुपर्छ (सीमाहरू र शैली)

तल, हामी स्पष्ट निर्देशनहरूसहितको यात्रा कन्सिएर्ज एजेन्ट बनाउँछौं जुन यसको प्रत्येक प्रतिक्रिया आकार दिन्छ।


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## ढाँचा २: पाइडान्टिक मोडेलहरू संग संरचित आउटपुट

स्वतन्त्र-रूप टेक्स्ट वार्तालापका लागि उपयोगी छ, तर डाउनस्ट्रीम प्रणालीहरूलाई संरचित डाटा आवश्यक हुन्छ।
**पाइडान्टिक मोडेलहरू** र **टूल फंक्शन** सँग जोड्दा, हामीले गर्न सक्छौं:

- एजेन्टको आउटपुटको लागि ठ्याक्कै स्किमा परिभाषित गर्ने
- प्रतिक्रियाहरू स्वचालित रूपमा मान्य गर्ने
- एजेन्टको नतिजाहरूलाई आवेदन तर्कमा भरपर्दो रूपमा एकीकृत गर्ने

लागू गर्ने प्रमुख कुरा हो एजेन्ट चलाउँदा `response_format` पास गर्नु। यसले मोडेललाई
स्वतन्त्र-रूप टेक्स्टको सट्टा मान्य गरिएको `TravelRecommendations` वस्तु ( जुन `response.value` मा उपलब्ध छ)
फिर्ता दिन बाध्य पार्छ। `get_destination_details` टूल पनि टाइप गरिएको
`DestinationRecommendation` फिर्ता दिन्छ, त्यसैले डाटा अन्त्यदेखि अन्त्यसम्म संरचित रहन्छ।


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## पात्रो ३: एकल जिम्मेवारी एजेन्टहरू

जटिल कार्यहरूले धेरै केन्द्रित एजेन्टहरूलाई काम विभाजन गर्दा लाभ उठाउँछन्, प्रत्येकको एकल जिम्मेवारी हुन्छ:

- एक **गन्तव्य विशेषज्ञ** जसले स्थानहरू र उपलब्धता बारे जान्दछ
- एक **लजिस्टिक्स योजनाकार** जसले उडानहरू, होटलहरू, र यात्रा कार्यक्रमहरू सम्हाल्दछ

यसले सफ्टवेयर इञ्जिनियरिङको *चिन्तनको विभाजन* सिद्धान्तलाई प्रतिबिम्बित गर्दछ — प्रत्येक एजेन्टलाई स्वतन्त्र रूपमा परीक्षण, मर्मत र सुधार गर्न सजिलो हुन्छ।


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

यस पाठमा हामीले यात्रा सिफारिस गर्ने परिदृश्यमा तीन एजेन्टिक डिजाइन ढाँचाहरू लागू गर्यौं:

| ढाँचा | मुख्य विचार | लाभ |
|---|---|---|
| **स्पष्ट निर्देशनहरू** | पर्सोना, जिम्मेवारीहरू, र बाधाहरू पहिल्यै परिभाषित गर्नुहोस् | सुसंगत, ब्रान्ड अनुसार एजेन्ट व्यवहार |
| **संरचित आउटपुट** | प्रतिक्रिया ढाँचाको रूपमा Pydantic मोडेलहरू प्रयोग गर्नुहोस् | प्रमाणित, मेसिन-पठनीय परिणामहरू |
| **एकल जिम्मेवारी** | प्रत्येक एजेन्टलाई एउटा केन्द्रित काम दिनुहोस् | परीक्षण, मर्मत, र संयोजन गर्न सजिलो |

यी ढाँचाहरू स्वाभाविक रूपमा संयोजन हुन्छन् — तपाईं स्पष्ट निर्देशनहरूलाई संरचित आउटपुटसँग एकल जिम्मेवारी एजेन्ट भित्र संयोजन गरेर बलियो, उत्पादन-तयार प्रणालीहरू बनाउन सक्नुहुन्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
